# Projektopgaver: Rundkorsel - Fully formatted CAS solution

This notebook follows a school-style CAS transcript with explicit assumptions and readable checks.
All trigonometric calculations use **degrees**.

## Given (from the map)

- Points: $A=(3,38)$, $B=(26,25)$, $P=(43,30)$
- Inner radius: $r_{in}=15$
- Outer radius: $r_{out}=25$
- Midline radius: $r_{mid}=20$
- Central angle: $\angle AOC=91^\circ$

In [ ]:
import sympy as sp
from IPython.display import Markdown, display

sp.init_printing(use_unicode=True)

# Symbols and constants
x, y, u, v, h, k = sp.symbols("x y u v h k", real=True)
pi = sp.pi

def N(expr, n=10):
    return sp.N(expr, n)

def deg(angle):
    return angle * pi / 180

def dist(P1, P2):
    return sp.sqrt((P1[0] - P2[0])**2 + (P1[1] - P2[1])**2)

def dot(v1, v2):
    return v1[0] * v2[0] + v1[1] * v2[1]

def rotate_deg(vec, angle_deg):
    c = sp.cos(deg(angle_deg))
    s = sp.sin(deg(angle_deg))
    return sp.Matrix([vec[0] * c - vec[1] * s, vec[0] * s + vec[1] * c])

def proj_on(v, d):
    return (dot(v, d) / dot(d, d)) * sp.Matrix(d)

def reflect_across_axis(v, d):
    return 2 * proj_on(v, d) - sp.Matrix(v)

A = sp.Matrix([3, 38])
B = sp.Matrix([26, 25])
P = sp.Matrix([43, 30])

r_in = sp.Integer(15)
r_out = sp.Integer(25)
r_mid = sp.Integer(20)

display(Markdown("## CAS helper setup complete"))
print("A =", A, "B =", B, "P =", P)
print("r_in =", r_in, "r_out =", r_out, "r_mid =", r_mid)

## 1) Inner circle equation (centered coordinates)

If the coordinate system is placed at the roundabout center, then:

- Inner circle: $u^2 + v^2 = 15^2 = 225$
- Outer circle: $u^2 + v^2 = 25^2 = 625$
- Midline circle: $u^2 + v^2 = 20^2 = 400$

In [ ]:
inner_eq = sp.Eq(u**2 + v**2, r_in**2)
outer_eq_centered = sp.Eq(u**2 + v**2, r_out**2)
mid_eq = sp.Eq(u**2 + v**2, r_mid**2)

display(Markdown("### Task 1 answer"))
sp.pprint(inner_eq)
print("Outer (centered):", outer_eq_centered)
print("Midline (centered):", mid_eq)

## 2) Outer circle equation through A and B

Solve

\[
(3-h)^2+(38-k)^2=625,\qquad (26-h)^2+(25-k)^2=625.
\]

There are two geometric center candidates; we choose the one with larger $y$ (matching the map).

In [ ]:
sol_centers = sp.solve(
    [sp.Eq((A[0] - h)**2 + (A[1] - k)**2, r_out**2),
     sp.Eq((B[0] - h)**2 + (B[1] - k)**2, r_out**2)],
    [h, k],
    dict=True,
)

print("CAS Out (exact centers):")
for i, sol in enumerate(sol_centers, start=1):
    print(f"O{i} = ({sp.simplify(sol[h])}, {sp.simplify(sol[k])})")

sol_chosen = max(sol_centers, key=lambda s: float(sp.N(s[k], 15)))
h0 = sp.simplify(sol_chosen[h])
k0 = sp.simplify(sol_chosen[k])
O = sp.Matrix([h0, k0])

print("\nChosen center (higher y):")
print("O ≈", (N(h0, 12), N(k0, 12)))

outer_eq = sp.Eq((x - h0)**2 + (y - k0)**2, r_out**2)
expanded_outer = sp.expand((x - h0)**2 + (y - k0)**2 - r_out**2)

display(Markdown("### Task 2 answer"))
sp.pprint(outer_eq)
print("Expanded form = 0:")
sp.pprint(expanded_outer)

## 3) One full lap time at 30 km/h (midline)

\[
L = 2\pi r_{mid}=40\pi,\qquad v=30\,\text{km/h}=\frac{30000}{3600}\,\text{m/s},\qquad T=\frac{L}{v}.
\]

In [ ]:
L = 2 * pi * r_mid
v_mps = sp.Rational(30000, 3600)
T = sp.simplify(L / v_mps)

display(Markdown("### Task 3 answer"))
print("L =", N(L, 12), "m")
print("T =", N(T, 12), "s")

## 4) Coordinates of C and D, then tangent angles

- $C$ is obtained by rotating $\overrightarrow{OA}$ by $+91^\circ$.
- $D$ is the reflection of $\overrightarrow{OB}$ across axis $OC$.

In [ ]:
vA = A - O
vB = B - O

vC_plus = rotate_deg(vA, 91)
vC_minus = rotate_deg(vA, -91)
C_plus = O + vC_plus
C_minus = O + vC_minus

# Choose the map-consistent C on the right/lower-right side
C = C_plus if float(sp.N(C_plus[0], 12)) > float(sp.N(C_minus[0], 12)) else C_minus
vC = C - O

vD = reflect_across_axis(vB, vC)
D = O + vD

print("C candidates:")
print("C+ ≈", (N(C_plus[0], 12), N(C_plus[1], 12)))
print("C- ≈", (N(C_minus[0], 12), N(C_minus[1], 12)))
print("Chosen C ≈", (N(C[0], 12), N(C[1], 12)))
print("D ≈", (N(D[0], 12), N(D[1], 12)))

check_C = sp.simplify((C[0] - h0)**2 + (C[1] - k0)**2 - r_out**2)
check_D = sp.simplify((D[0] - h0)**2 + (D[1] - k0)**2 - r_out**2)
print("Check C on outer circle ->", check_C)
print("Check D on outer circle ->", check_D)

phi_AC = sp.Integer(180) - sp.Integer(91)
theta_BD = sp.acos(sp.simplify(dot(vB, vD) / r_out**2)) * 180 / pi
phi_BD = sp.simplify(180 - theta_BD)

display(Markdown("### Task 4 answers"))
print("Angle between tangents at A and C =", phi_AC, "deg")
print("Angle between tangents at B and D ≈", N(phi_BD, 12), "deg")

## 5) Shortest distance from midline-driver to blue car center $P(43,30)$

Driver moves on the midline circle centered at $O$ with radius $r_{mid}=20$.
Shortest distance from point $P$ to that circle:

\[
d_{min} = \left|\,|OP| - r_{mid}\right|.
\]

In [ ]:
OP = dist(P, O)
d_min = sp.Abs(OP - r_mid)

display(Markdown("### Task 5 answer"))
print("|OP| ≈", N(OP, 12), "m")
print("d_min ≈", N(d_min, 12), "m")

## 6) New road from northeast, arriving like Kalundborgvej

Assumption used:
- New centerline comes from NE direction ($45^\circ$) through $O$.
- It has the same half-width angle as Kalundborgvej approach.

Let $\alpha = \angle BOC$.
Then centerline hit point on the outer circle is
\[
E = O + r_{out}(\cos 45^\circ,\sin 45^\circ).
\]
The two edge contact points are
\[
E_1 = O + r_{out}\,\mathrm{Rotate}(\hat u,+\alpha),\qquad
E_2 = O + r_{out}\,\mathrm{Rotate}(\hat u,-\alpha),
\]
where $\hat u=(\cos45^\circ,\sin45^\circ)$.

In [ ]:
alpha = sp.acos(sp.simplify(dot(vB, vC) / r_out**2)) * 180 / pi
u_NE = sp.Matrix([sp.cos(deg(45)), sp.sin(deg(45))])

# Centerline y = x + (k0 - h0)
b_NE = sp.simplify(k0 - h0)

E = O + r_out * u_NE
E1 = O + r_out * rotate_deg(u_NE, alpha)
E2 = O + r_out * rotate_deg(u_NE, -alpha)

def tangent_standard(T):
    a = sp.simplify(T[0] - h0)
    b = sp.simplify(T[1] - k0)
    rhs = sp.simplify(r_out**2 + a*h0 + b*k0)
    return a, b, rhs

a1, b1, rhs1 = tangent_standard(E1)
a2, b2, rhs2 = tangent_standard(E2)

display(Markdown("### Task 6 answers"))
print("Centerline: y = x +", N(b_NE, 12))
print("alpha ≈", N(alpha, 12), "deg")
print("E ≈", (N(E[0], 12), N(E[1], 12)))
print("E1 ≈", (N(E1[0], 12), N(E1[1], 12)))
print("E2 ≈", (N(E2[0], 12), N(E2[1], 12)))

print("\nTangent line 1: a1*x + b1*y = rhs1")
print(N(a1, 12), "*x +", N(b1, 12), "*y =", N(rhs1, 12))

print("Tangent line 2: a2*x + b2*y = rhs2")
print(N(a2, 12), "*x +", N(b2, 12), "*y =", N(rhs2, 12))

## Final numeric summary (clean)

This table-like block matches the final answers from the CAS transcript.

In [ ]:
summary = {
    "Outer center O": (N(h0, 8), N(k0, 8)),
    "Outer circle": f"(x-{N(h0, 8)})^2 + (y-{N(k0, 8)})^2 = 625",
    "Inner centered circle": "u^2 + v^2 = 225",
    "Lap length L (m)": N(L, 8),
    "Lap time T (s)": N(T, 8),
    "Point C": (N(C[0], 8), N(C[1], 8)),
    "Point D": (N(D[0], 8), N(D[1], 8)),
    "Tangents angle at A,C (deg)": phi_AC,
    "Tangents angle at B,D (deg)": N(phi_BD, 8),
    "Shortest distance to P (m)": N(d_min, 8),
    "New centerline": f"y = x + {N(b_NE, 8)}",
    "Hit point E": (N(E[0], 8), N(E[1], 8)),
    "Tangency point E1": (N(E1[0], 8), N(E1[1], 8)),
    "Tangency point E2": (N(E2[0], 8), N(E2[1], 8)),
    "Tangent line 1": f"{N(a1, 8)}*x + {N(b1, 8)}*y = {N(rhs1, 8)}",
    "Tangent line 2": f"{N(a2, 8)}*x + {N(b2, 8)}*y = {N(rhs2, 8)}",
}

for key, value in summary.items():
    print(f"- {key}: {value}")